In [2]:
import pandas as pd

# 1. ファイル読み込み
type_df = pd.read_csv("2025_成績_タイプ.csv")
runs_df = pd.read_csv("4番打者平均得点.csv")  
# もし別名で保存しているなら、ここを
# "npb_2025_main_fourth_batter_avg_runs_when_batting_4th.csv"
# に変えてください

# 2. 選手名の空白を整える
type_df["選手名"] = type_df["選手名"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
runs_df["選手名"] = runs_df["選手名"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 3. 結合
merged_df = pd.merge(
    type_df,
    runs_df,
    on=["球団", "選手名"],
    how="left"
)

# 4. 必要列だけ確認
cols = [c for c in [
    "球団", "選手名", "4番タイプ",
    "本塁打", "OBP", "SLG", "OPS",
    "4番試合数", "合計得点", "平均得点"
] if c in merged_df.columns]

merged_df = merged_df[cols].copy()

# 5. タイプ別平均
type_summary = (
    merged_df.groupby("4番タイプ", as_index=False)
    .agg({
        "平均得点": "mean",
        "4番試合数": "mean",
        "本塁打": "mean",
        "OBP": "mean",
        "SLG": "mean",
        "OPS": "mean"
    })
)

# 6. 小数整理
for col in ["平均得点", "4番試合数", "本塁打", "OBP", "SLG", "OPS"]:
    if col in type_summary.columns:
        type_summary[col] = type_summary[col].round(3)

# 7. 保存
merged_df.to_csv("4番タイプ_平均得点_結合表.csv", index=False, encoding="utf-8-sig")
type_summary.to_csv("4番タイプ別比較表.csv", index=False, encoding="utf-8-sig")

# 8. 表示
print("=== 選手ごとの結合表 ===")
print(merged_df)

print("\n=== 4番タイプ別比較表 ===")
print(type_summary)

print("\n保存完了:")
print("- 4番タイプ_平均得点_結合表.csv")
print("- 4番タイプ別比較表.csv")

=== 選手ごとの結合表 ===
        球団     選手名 4番タイプ  本塁打    OBP    SLG    OPS  4番試合数   合計得点   平均得点
0     DeNA    牧 秀悟   総合型   16  0.325  0.475  0.800     58  155.0  2.672
1    オリックス  杉本 裕太郎   総合型   16  0.332  0.426  0.758     75  290.0  3.867
2   ソフトバンク   山川 穂高   長打型   23  0.300  0.402  0.702     68  262.0  3.853
3     ヤクルト     オスナ   長打型   14  0.307  0.377  0.684     63  180.0  2.857
4      ロッテ   山本 大斗   長打型   11  0.262  0.338  0.600     47  156.0  3.319
5       中日   細川 成也   総合型   20  0.367  0.489  0.856     75  236.0  3.147
6       巨人   岡本 和真   総合型   15  0.416  0.598  1.014     66  238.0  3.606
7       広島   末包 昇大   長打型   11  0.296  0.373  0.669     61  194.0  3.180
8     日本ハム   野村 佑希   総合型    8  0.325  0.398  0.723     52  175.0  3.365
9       楽天     ボイト   総合型   13  0.384  0.498  0.882     35  120.0  3.429
10      西武     ネビン   長打型   21  0.346  0.448  0.794    117  326.0  2.786
11      阪神   佐藤 輝明   長打型   40  0.345  0.579  0.924    124  431.0  3.476

=== 4番タイプ別比較表 ===
  4番タイプ   平均得点   4番試合数     本

In [3]:
if len(type_summary) == 2:
    type_summary = type_summary.sort_values("平均得点", ascending=False).reset_index(drop=True)
    top_type = type_summary.loc[0, "4番タイプ"]
    top_score = type_summary.loc[0, "平均得点"]
    second_type = type_summary.loc[1, "4番タイプ"]
    second_score = type_summary.loc[1, "平均得点"]

    print(f"\n平均得点が高かったのは {top_type} で、{top_score} 点でした。")
    print(f"{second_type} は {second_score} 点でした。")
    print(f"差は {round(top_score - second_score, 3)} 点です。")


平均得点が高かったのは 総合型 で、3.348 点でした。
長打型 は 3.245 点でした。
差は 0.103 点です。


1. 分析方法に書くこと

ここでは、何をどうやって調べたか を書きます。

書く内容は例えばこれです。
	•	対象はNPB2025年12球団
	•	各球団で最も4番打者として出場した選手を代表4番打者とした
	•	代表4番打者の打撃成績を取得した
	•	本塁打・SLG・OPSを重視した長打型スコア、OBP・SLG・OPSを重視した総合型スコアを作成した
	•	そのスコアに基づいて長打型 / 総合型に分類した
	•	さらに、その選手が4番で出場した試合のチーム得点を集計し、平均得点を比較した

つまり、手順を書く場所です。

⸻

2. 研究結果に書くこと

ここでは、出た数字をそのまま書く のが中心です。

今回なら特に大事なのはここです。
	•	総合型4番の平均得点は 3.348点
	•	長打型4番の平均得点は 3.245点
	•	差は 0.103点
	•	総合型の方がやや高かった

ここでは、あまり解釈しすぎず、事実を書く のが大事です。

⸻

3. 考察に書くこと

ここでは、なぜそうなったのかを自分なりに説明する 部分です。

たとえば書けるのは次です。
	•	4番打者には本塁打数だけでなく、出塁を含めた総合的な打撃力が重要である可能性がある
	•	一発の長打力だけでなく、凡退しにくさや走者を返すための安定性も得点につながった可能性がある
	•	ただし差は0.103点と小さいため、圧倒的な差とまでは言えない
	•	チーム得点は4番以外の打者や相手投手の影響も受けるため、4番だけで全てを説明できるわけではない

つまり、結果の意味づけを書く場所です。

⸻

4. 今後の課題に書くこと

ここでは、今回できなかったこと・今後広げたいこと を書きます。

今のあなたなら、これが書きやすいです。
	•	今回はNPBのみを対象にした
	•	今後はMLBにも同様の分析を広げたい
	•	今回は通算成績ベースで分類したが、今後は4番時成績のみで比較したい
	•	打点や得点圏成績も含めると、より詳しい4番像が見える可能性がある


1. 分析方法

本研究では、NPB2025年の12球団を対象に、各球団で最も4番打者として出場した選手を代表4番打者と定義した。まず、球団ごとのスタメン一覧をスクレイピングし、各試合の4番打者を抽出した。その後、各球団で最も多く4番として出場した選手を集計し、代表4番打者として選定した。

次に、代表4番打者の打撃成績を取得し、長打力を重視する「長打型」と、出塁を含めた総合打撃力を重視する「総合型」に分類した。分類にあたっては、本塁打・長打率・OPSを用いた長打型スコアと、出塁率・長打率・OPSを用いた総合型スコアを作成した。各指標は単位が異なるため、比較可能にするためにzスコアで標準化した。

さらに、各代表4番打者が実際に4番で出場した試合のみを対象に、その試合でチームが記録した得点を集計した。そして、4番タイプごとに平均得点を算出し、長打型4番と総合型4番で比較を行った。

2. 研究結果

分析の結果、代表4番打者が4番で出場した試合におけるチーム平均得点は、総合型4番の方が長打型4番よりもやや高かった。具体的には、総合型4番の平均得点は 3.348点、長打型4番の平均得点は 3.245点 であり、その差は 0.103点 であった。

この結果から、少なくともNPB2025年の12球団を対象とした今回の分析では、4番打者を本塁打数の多さだけで評価するよりも、出塁率やOPSを含めた総合打撃力に注目した方が、得点との関係を捉えやすい可能性があることが分かった。

3. 考察

本研究では、4番打者というと本塁打数や長打力が重視されやすいと考えられるが、実際の分析では、長打型よりも総合型の方が平均得点がわずかに高いという結果になった。これは、4番打者には一発の長打力だけでなく、出塁して攻撃機会を広げる力や、安定して得点に絡む総合的な打撃力も重要である可能性を示している。

特に、4番打者は得点圏で打席が回る場面が多いため、単に本塁打を打てることだけでなく、四球を選べることや安打によって走者を返せることも得点に結びつくと考えられる。そのため、出塁率やOPSを含めて評価した総合型4番が、やや高い平均得点を示したのは自然な結果ともいえる。

ただし、差は 0.103点 と小さく、長打型と総合型に大きな開きがあるとまでは言えない。また、チーム得点は4番打者だけで決まるものではなく、1〜3番の出塁、下位打線のつながり、相手投手、球場条件などの影響も受ける。そのため、本研究の結果は「総合型4番の方がやや高い傾向が見られた」と解釈するのが適切である。

4. 今後の課題

本研究では、NPB2025年の代表4番打者を対象として分析を行ったが、今後はさらに分析対象や指標を広げる必要がある。例えば、今回は代表4番打者のシーズン成績をもとにタイプ分類を行ったが、今後は「4番として出場した試合のみの打撃成績」を用いて分類すれば、より4番の役割に即した分析が可能になると考えられる。

また、今回はNPBのみを対象としたが、今後は同様の手法をMLBにも適用し、4番打者に求められる能力がリーグによって異なるのかを比較することも課題である。さらに、得点だけでなく、打点や得点圏成績、併殺打などの指標も加えることで、4番打者に必要な能力をより詳しく明らかにできる可能性がある。
